# 04 — Barchart Premier WTI futures curve

Source files are manually exported from Barchart Premier
Interactive Chart using expressions:

- `{CL*1}`
- `{CL*2}`
- `{CL*3}`
- `{CL*4}`
- `{CL*5}`
- `{CL*6}`
- `{CL*12}`

with frequency **Daily Nearby**.

Files must be stored in:

`data/external/barchart/`

with names:

- `cl_f1_daily_nearby.csv`
- `cl_f2_daily_nearby.csv`
- `cl_f3_daily_nearby.csv`
- `cl_f4_daily_nearby.csv`
- `cl_f5_daily_nearby.csv`
- `cl_f6_daily_nearby.csv`
- `cl_f12_daily_nearby.csv`

Barchart can already include the next Globex trading date while the
download timestamp is still on the prior calendar day. The loader drops
that unfinished row automatically.

The exported `Study: VOL (LikeMain)` is not maturity-specific and Open
Interest is unavailable in these chart exports, so neither is used as a
futures-curve model feature.


In [2]:
from pathlib import Path

import pandas as pd

import sys
sys.path.append('../../')

from src.data.barchart_premier_loader import (
    EXPECTED_NEARBYS,
    load_barchart_wti_curve,
)


def find_project_root(
    start: Path | None = None,
) -> Path:
    current = (start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate

    msg = "Could not locate project root."
    raise FileNotFoundError(msg)


ROOT = find_project_root()

SOURCE_DIR = (
    ROOT / "data" / "external" / "barchart"
)
PROCESSED_DIR = (
    ROOT / "data" / "processed" / "market"
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print("source:", SOURCE_DIR)
print("expected nearbys:", EXPECTED_NEARBYS)


source: /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/external/barchart
expected nearbys: (1, 2, 3, 4, 5, 6, 12)


## 1. Parse and validate all seven nearby series


In [3]:
barchart_long, barchart_curve, barchart_metadata = (
    load_barchart_wti_curve(
        SOURCE_DIR,
        start_date="2015-01-01",
    )
)

display(barchart_metadata)

print(
    "model date range:",
    barchart_curve["trading_day"].min(),
    "->",
    barchart_curve["trading_day"].max(),
)
print(
    "daily curve rows:",
    len(barchart_curve),
)
print(
    "long rows:",
    len(barchart_long),
)


,nearby,expression,raw_filename,downloaded_at,download_timezone,raw_first_day,raw_last_day,raw_rows,model_rows,open_interest_available
0,1,{CL*1},cl_f1_daily_nearby.csv,2026-11-05 17:46:00,CST,2000-01-04,2026-11-05,6743,2981,False
1,2,{CL*2},cl_f2_daily_nearby.csv,2026-11-05 17:50:00,CST,2000-01-04,2026-11-05,6743,2981,False
2,3,{CL*3},cl_f3_daily_nearby.csv,2026-11-05 17:51:00,CST,2000-01-04,2026-11-05,6743,2981,False
3,4,{CL*4},cl_f4_daily_nearby.csv,2026-11-05 17:51:00,CST,2000-01-04,2026-11-05,6742,2981,False
4,5,{CL*5},cl_f5_daily_nearby.csv,2026-11-05 17:52:00,CST,2000-01-04,2026-11-05,6742,2981,False
5,6,{CL*6},cl_f6_daily_nearby.csv,2026-11-05 17:46:00,CST,2000-01-04,2026-11-05,6742,2981,False
6,12,{CL*12},cl_f12_daily_nearby.csv,2026-11-05 17:52:00,CST,2000-01-04,2026-11-05,6738,2981,False


model date range: 2015-01-02 00:00:00 -> 2026-11-05 00:00:00
daily curve rows: 2981
long rows: 20867


## 2. Per-nearby QA


In [4]:
nearby_qa = (
    barchart_long.groupby("nearby")
    .agg(
        rows=("trading_day", "size"),
        first_day=("trading_day", "min"),
        last_day=("trading_day", "max"),
        missing_open=("open", lambda s: s.isna().sum()),
        missing_high=("high", lambda s: s.isna().sum()),
        missing_low=("low", lambda s: s.isna().sum()),
        missing_close=("close", lambda s: s.isna().sum()),
    )
)

display(nearby_qa)

assert nearby_qa["rows"].nunique() == 1
assert nearby_qa["missing_close"].sum() == 0


,rows,first_day,last_day,missing_open,missing_high,missing_low,missing_close
nearby,,,,,,,
1,2981,2015-01-02,2026-11-05,0,0,0,0
2,2981,2015-01-02,2026-11-05,0,0,0,0
3,2981,2015-01-02,2026-11-05,0,0,0,0
4,2981,2015-01-02,2026-11-05,0,0,0,0
5,2981,2015-01-02,2026-11-05,0,0,0,0
6,2981,2015-01-02,2026-11-05,0,0,0,0
12,2981,2015-01-02,2026-11-05,0,0,0,0


## 3. Inspect the WTI curve and calendar spreads


In [5]:
display(
    barchart_curve[
        [
            "trading_day",
            "wti_f1",
            "wti_f2",
            "wti_f3",
            "wti_f4",
            "wti_f5",
            "wti_f6",
            "wti_f12",
            "spread_f1_f2",
            "spread_f2_f3",
            "spread_f1_f3",
            "spread_f1_f6",
            "spread_f1_f12",
        ]
    ].tail(10)
)


nearby,trading_day,wti_f1,wti_f2,wti_f3,wti_f4,wti_f5,wti_f6,wti_f12,spread_f1_f2,spread_f2_f3,spread_f1_f3,spread_f1_f6,spread_f1_f12
2971,2026-10-23,103.27,100.94,98.36,95.68,93.2,91.02,81.54,2.33,2.58,4.91,12.25,21.73
2972,2026-10-26,101.76,99.49,97.17,94.83,92.67,90.74,81.98,2.27,2.32,4.59,11.02,19.78
2973,2026-10-27,99.9,98.06,96.14,94.17,92.28,90.55,82.07,1.84,1.92,3.76,9.35,17.83
2974,2026-10-28,102.78,100.99,98.99,96.85,94.77,92.82,83.15,1.79,2.0,3.79,9.96,19.63
2975,2026-10-29,102.0,100.14,98.09,95.88,93.74,91.77,82.54,1.86,2.05,3.91,10.23,19.46
2976,2026-10-30,104.12,102.21,99.94,97.49,95.12,92.96,83.01,1.91,2.27,4.18,11.16,21.11
2977,2026-11-02,96.46,94.8,92.99,91.13,89.4,87.84,80.05,1.66,1.81,3.47,8.62,16.41
2978,2026-11-03,96.09,94.77,93.0,91.14,89.43,87.9,80.31,1.32,1.77,3.09,8.19,15.78
2979,2026-11-04,88.12,87.1,85.76,84.27,82.85,81.57,75.32,1.02,1.34,2.36,6.55,12.8
2980,2026-11-05,90.52,89.2,87.39,85.44,83.63,82.05,75.11,1.32,1.81,3.13,8.47,15.41


Positive `F1-F2` means the front of the curve is backwardated.
Negative `F1-F2` means contango.


In [6]:
spread_summary = barchart_curve[
    [
        "spread_f1_f2",
        "spread_f2_f3",
        "spread_f1_f3",
        "spread_f1_f6",
        "spread_f1_f12",
    ]
].describe().T

display(spread_summary)


,count,mean,std,min,25%,50%,75%,max
nearby,,,,,,,,
spread_f1_f2,2981.0,0.199628,1.232947,-7.29,-0.31,0.07,0.51,13.94
spread_f2_f3,2981.0,0.256817,1.061366,-4.47,-0.3,0.16,0.51,8.65
spread_f1_f3,2981.0,0.456444,2.273873,-11.76,-0.61,0.25,1.02,22.17
spread_f1_f6,2981.0,1.233901,4.50801,-15.07,-1.37,0.95,2.53,36.7
spread_f1_f12,2981.0,2.5552,6.844894,-18.55,-1.95,2.52,5.12,41.57


## 4. Final completeness QA


In [7]:
price_columns = [
    "wti_f1",
    "wti_f2",
    "wti_f3",
    "wti_f4",
    "wti_f5",
    "wti_f6",
    "wti_f12",
]

complete_curve_share = (
    barchart_curve[price_columns]
    .notna()
    .all(axis=1)
    .mean()
)

assert not barchart_curve[
    "trading_day"
].duplicated().any()

assert complete_curve_share == 1.0

print(
    "complete F1-F6/F12 share:",
    round(complete_curve_share, 4),
)
print("Barchart curve QA passed.")


complete F1-F6/F12 share: 1.0
Barchart curve QA passed.


## 5. Save processed source layer


In [8]:
long_path = (
    PROCESSED_DIR
    / "barchart_wti_nearbys_2015_onward.parquet"
)
curve_path = (
    PROCESSED_DIR
    / "barchart_wti_curve_2015_onward.parquet"
)
metadata_path = (
    PROCESSED_DIR
    / "barchart_wti_download_metadata.parquet"
)

barchart_long.to_parquet(
    long_path,
    index=False,
)
barchart_curve.to_parquet(
    curve_path,
    index=False,
)
barchart_metadata.to_parquet(
    metadata_path,
    index=False,
)

print("saved:", long_path)
print("saved:", curve_path)
print("saved:", metadata_path)


saved: /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/processed/market/barchart_wti_nearbys_2015_onward.parquet
saved: /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/processed/market/barchart_wti_curve_2015_onward.parquet
saved: /Users/ivanandreev/Desktop/ИИ/Нефть/oil-news-graph/data/processed/market/barchart_wti_download_metadata.parquet


## 6. Final QA


In [9]:
expected_outputs = [
    long_path,
    curve_path,
    metadata_path,
]

for path in expected_outputs:
    assert path.exists(), path
    print(path.name, ": OK")

print(
    "Barchart Premier WTI source layer completed."
)


barchart_wti_nearbys_2015_onward.parquet : OK
barchart_wti_curve_2015_onward.parquet : OK
barchart_wti_download_metadata.parquet : OK
Barchart Premier WTI source layer completed.


## Next step — EIA validation

Compare Barchart `F1–F4` with the existing EIA `RCLC1–RCLC4` series
over their common 2015–2024 period.

Barchart Premier `Daily Nearby` uses its own liquidity-based nearby roll,
so the two sources are not expected to match exactly around roll dates.
We should compare MAE, median absolute difference, correlation, and
differences around roll windows before selecting the final modelling
convention.
